# Class 4 - Tabular Data

Today, we move from NumPy arrays to Pandas tables (using the time-series from Future Forests Hartheim).

Goals:
- use Pandas to analyze time-series tabular data (pandas operations and resample),
- make a plot with Matplotlib.


## 1. From NumPy arrays to Pandas tables

Now we are finally going to move from these dummy exercises and datasets to real-world data. First, we will use the Future Forests Hartheim climate station file in: 

`../processed_data/26030350_hourly.csv`

Note: We do not work directly with the raw data file, but rather with a cleaned, post-processed version containing hourly data, to make the work easier.

In addition, we are going to use the library `pandas` instead of `NumPy` because `pandas` provides convenient and more powerfull tools for working with structured, time-series datasets, such as selecting columns, filtering data, handling missing values, and much more.

In [ ]:
import pandas as pd


### i. Loading a CSV with `pd.read_csv(...)`

`pd.read_csv(...)` reads a CSV file and returns a `DataFrame`, the central object in Pandas and a new variable type for us. A `DataFrame` is a table with column and row indices. You can think of it as an Excel table inside Python.

In [ ]:
HartheimRaw = pd.read_csv("../processed_data/26030350_hourly.csv", parse_dates=["utc_time"])
print(type(HartheimRaw))

After importing our DataFrame, it is always useful to print some information. 

> **Note about `Pandas` use:** we don't use a function in the same way as with `NumPy`. Instead of `np.shape(Array)`, we do `DataFrame.shape`:

In [ ]:
print(HartheimRaw.shape)   # shape of the data frame

In [ ]:
print(HartheimRaw.columns) # all columns

In [ ]:
print(HartheimRaw.head(2)) # the first two rows

Above, the row index is simply a number, but you can choose something else. The function `set_index("utc_time")` allows you to set a column as the index instead. Those row labels are then available as `DataFrame.index` (here, the UTC times).

In [ ]:
HartheimRaw = HartheimRaw.set_index("utc_time")
print(HartheimRaw.head(2))

As you can see we moved from data with less than 10 data points to a dataset with 330*22 = 7.260 entries.

### ii. Basic Pandas operations

#### a. Column selection

A DataFrame often has more columns than you need. You can select a subset by passing a list of column names.


In [ ]:
HartheimData = HartheimRaw[["temperature_2m", "humidity_2m", "temperature_ground", "precipitation"]].copy() # .copy() is really important
print(HartheimData.columns)

#### b. Indexing and slicing

In `NumPy`, indexing is always by **position**: `Array[0]` is the first element, and `Array[1:5]` selects second to fifth element.

In `pandas`, there are two ways to do indexing and slicing:

- by **position** (row/column numbers) with `.iloc[row_index, column_index]` (similar to `Numpy`):


In [ ]:
print(HartheimData.iloc[0, 1]) # First row, second column

In [ ]:
print(HartheimData.iloc[0:3, 1:3]) # First 3 rows, 2 -> 3 columns


- or by **label** (names / dates) with `.loc[row_names, column_names]`. This is much more readable than using number indices:


In [ ]:
# Select one row, one column
print(HartheimData.loc["2026-07-25 12:00:00", "temperature_2m"])

In [ ]:
# Slice the rows and select two columns
print(HartheimData.loc["2026-07-24":"2026-07-26", ["temperature_2m", "temperature_ground"]])

> Note: with `.loc`, a slice like `'2026-07-24':'2026-07-26'` is **inclusive on both ends**. In `NumPy`, the stop index is excluded.

Try to select the `humidity_2m` values of the first two days in August: 

#### c. Vectorisation operations with Pandas

Like in NumPy, you can use vectorised math and statistical operations on Pandas columns:

In [ ]:
MeanHumidity = HartheimData["humidity_2m"].mean()
print("Mean 2m humidity (in %):", MeanHumidity)

Try calculating the total precipitation for the last two days of July in a single line

#### d. Creating a new column

You can create a new column by assigning the result of an operation between existing columns. Here we compute the vertical temperature gradient (2 m minus ground). The operation is vectorised, just like in NumPy.


In [ ]:
HartheimData["temp_gradient_2m_minus_ground"] = HartheimData["temperature_2m"] - HartheimData["temperature_ground"]
print(HartheimData.loc["2026-07-25 12:00:00", ["temperature_2m", "temperature_ground", "temp_gradient_2m_minus_ground"]])

Try to add a new `deficit_humidity_2m` column (`Deficity Humidity = 100 - humidity_2m`)

## Exercise A

This exercise is a copy of **Class 3 Exercise A**, except that you must use the Hartheim DataFrame.

Your American cousin, an avid gardener, comes to visit you and doesn't understand degrees Celsius. The Hartheim station records air temperature at 2 m in °C (`temperature_2m`), but your cousin wants to read it in °F. Give the average in both Celsius and Fahreneit.

#### e. Filtering rows

Filtering keeps only the rows that match a condition. In Pandas this uses a **Boolean mask**, in this case a column of `True`/`False` values, one per row. You write the condition, then put the mask inside `[]`.

In [ ]:
HotTemp = HartheimData[HartheimData["temperature_2m"] > 38]
print("Hot temperatures (>38°C):", HotTemp["temperature_2m"])

If you only need the matching row labels (e.g. the UTC times), use `DataFrame.index[mask]` instead of `DataFrame[mask]`:

In [ ]:
print(HartheimData.index[HartheimData["temperature_2m"] > 38])

Try to print the number of hours with cold temperatures (colder than 10°C)

Try to print the numbers of hours which are warm (`temperature_2m > 25`) and humid (`humidity_2m > 60`):

### iii. Changing time resolution (`resample`)

Sometimes we want to change the time resolution of a series. For example, by aggregating hourly measurements into daily values. 

**Resampling** groups rows into coarser time bins (e.g. days) and then applies an aggregation such as mean or sum.

Because the index is already a datetime, you can write:

- `.resample('D').mean()` — daily average (good for temperature, humidity)
- `.resample('D').sum()` — daily total (good for precipitation)

`'D'` is a day frequency. Other common frequencies: `'h'` (hour), `'W'` (week), `'ME'` (month) or `'YE'` (year).

In [ ]:
DailyMean = HartheimData[["temperature_2m", "humidity_2m"]].resample("D").mean()
print("Before:")
print(HartheimData[["temperature_2m", "humidity_2m"]])
print("After:")
print(DailyMean)

In [ ]:
DailyPrecip = HartheimData[["precipitation"]].resample("D").sum()
print(DailyPrecip)

Try to print the weekly mean temperature and total precipitation of the whole time-series

## Exercise B

Root growth and soil microbial activity often depend on how warm the upper soil layers are compared with deeper layers. You are investigating whether the upper soil layer becomes substantially warmer than the deeper soil layer during warm conditions at the Hartheim station.

The logger measures soil temperature at two depths: about 0.2 m (`soil_temperature_0_2`) and about 0.6 m (`soil_temperature_0_6`).

Tasks:
1. Compute the mean and standard deviation of both soil temperature columns (`soil_temperature_0_2`, `soil_temperature_0_6`). Look up online for how to use standard deviation in `Pandas`
2. For how many hours is the 0.2 m temperature higher than the 0.6 m temperature, and for how many hours is the opposite true? 
3. Print the UTC times of all hours corresponding to the condition that occurs for the fewer number of hours (hint: use `DataFrame.index[mask]`).

## 2. Plotting: Matplotlib essentials

Matplotlib is the most widely used plotting library in Python.

In [ ]:
import matplotlib.pyplot as plt

Start with a plot type function:

- `plt.plot(x, y, label='...')`, draws a line from the x and y columns. The `label` is used by the legend. 

Then you have hundreds of different visual options to choose. Here are the main one:
- `plt.title('...')` — sets the plot title.
- `plt.xlabel('...')` / `plt.ylabel('...')` — set the axis labels.
- `plt.legend()` — displays a legend using the labels defined in each `plt.plot()` call.

Finally, you can display the figure with `plt.show()`.

In [ ]:
plt.plot(HartheimData.index, HartheimData["temperature_2m"], label="Temperature 2m")
# You can put a second line
plt.plot(HartheimData.index, HartheimData["temperature_ground"], label="Temperature ground")

plt.title("Hourly temperatures (2026-07-24 to 2026-07-27)")
plt.xlabel("UTC time")
plt.ylabel("Temperature (°C)")
plt.legend()

plt.xticks(rotation=45)

plt.show()

Try to plot the humidity of the time-series

This course will mostly use line plots. Look on the online documentation to see all the different plot types you can make: https://matplotlib.org/stable/plot_types/index.html

## Exercise C

Using the time-series from Hartheim, you want to check if the hottest day is also the most physiologically stressful one for the trees. For trees, "stressful" usually means weather conditions that forces stomata to close.

Stomata do not shut only because the air is hot, but because it is dry. The relevant variable is the vapour pressure deficit (VPD): the difference between how much water vapour the air could hold and how much it actually holds.

You can compute VPD from the station data with the Tetens formula:

$$e_s = 0.6108 \cdot \exp\left(\frac{17.27\,T}{T + 237.3}\right) \qquad \text{VPD} = e_s \cdot \left(1 - \frac{RH}{100}\right)$$

with $T$ the air temperature in °C, $RH$ the relative humidity in %, and $e_s$ and VPD in kPa.

Tasks:

1. Using `temperature_2m` and `humidity_2m`, create a new column `vpd_kpa`. You will need `np.exp()`

2. Print the mean, minimum and maximum of `vpd_kpa`, and the UTC time at which the maximum occurs

3. A VPD above 3 kPa is often used as an indicative threshold above which stomatal conductance in temperate conifers is strongly reduced. How many hours of the record are above it? Plot VPD, and add a horizontal line at 3 kPa (look up `plt.axhline` in the Matplotlib documentation). In another cell, plot the temperature.

4. Compare the two rankings: is the day with the highest mean temperature also the day with the highest maximum VPD? Use resampling method to convert hours into days